# Minimax (game playing)

_Artificial Intelligence (CS221)_

**You play to win, your opponent plays to beat you. Plan for their best reply.**

In a two-player game, you and an opponent take turns. You want a high score. They want it low.

     Minimax assumes the opponent plays perfectly against you.

---

This notebook builds the minimax algorithm a piece at a time — first on a tiny abstract game tree, then on a real tic-tac-toe position. Run each cell, read the short note above it, and watch how the value at each node is decided by whose turn it is. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — Python

We start with a game as a **tree**: leaves are final scores (utilities), and internal nodes are turns. We build it in three steps: (1) the recursive minimax rule, (2) the tree to run it on, (3) reading the value off each subtree and the root.

### Step 1 — The minimax recursion

Minimax assumes both players are perfect: the **MAX** player picks the child with the highest value, the **MIN** player (the opponent) picks the lowest. The function recurses down the tree, flipping `maximizing` at each level because the turn alternates. A leaf is just a number, so it returns its own utility — that is the base case that stops the recursion.

In [ ]:
def minimax(node, maximizing):
    # Base case: a leaf is a plain number — return its utility directly.
    if isinstance(node, int):
        return node

    # Recurse into each child, flipping whose turn it is one level down.
    values = [minimax(child, not maximizing) for child in node]

    # MAX takes the best for itself; MIN (the opponent) takes the worst for us.
    if maximizing:
        return max(values)
    else:
        return min(values)

### Step 2 — Define the game tree

We encode the tree with nested lists: a number is a leaf utility, and a list is an internal node whose elements are its children. This tree has three levels of nodes above the leaves — the root is a **MAX** node, its children are **MIN** nodes, and their children are **MAX** nodes whose children are the leaf scores.

In [ ]:
# Nested lists = game tree. A number is a leaf utility; a list is an internal node.
tree = [[[3, 12], [8, 2]], [[4, 6], [14, 5]]]

### Step 3 — Evaluate the subtrees and the root

The root is a **MAX** node, so we call `minimax` with `maximizing=True`. To make the reasoning visible, we also print the value of each immediate subtree, which the opponent controls as a **MIN** node. The root then picks the larger of those two subtree values — that final number is the score MAX can guarantee against a perfect opponent.

In [ ]:
# Root is a MAX node; layers alternate MAX, MIN, MAX, then leaves.
root_value = minimax(tree, maximizing=True)

print("MAX over two MIN subtrees")
for i, sub in enumerate(tree):
    subtree_value = minimax(sub, maximizing=False)
    print("  subtree", i, "MIN value =", subtree_value)

print("minimax value at root:", root_value)

## Visualize the data & results

_In a tic-tac-toe position where O threatens to win, which square should X play?_

Now the game tree is a real tic-tac-toe board. We score each candidate move with minimax, then plot the values so the best defensive move stands out.

### Step 1 — Tic-tac-toe rules and full-board minimax

`winner` scans the eight winning lines and reports `"X"`, `"O"`, or `None`. `minimax` then plays the game out to the end: a finished board scores `+1` if X wins, `-1` if O wins, and `0` for a draw. For unfinished boards it tries every empty square, recursing with the turn handed to the other player — X maximizes the score, O minimizes it.

In [ ]:
def winner(b):
    lines = [(0, 1, 2), (3, 4, 5), (6, 7, 8),
             (0, 3, 6), (1, 4, 7), (2, 5, 8),
             (0, 4, 8), (2, 4, 6)]
    for a, c, d in lines:
        if b[a] != " " and b[a] == b[c] == b[d]:
            return b[a]
    return None

def minimax(b, player):
    # Terminal scores: X win = +1, O win = -1, full board with no winner = draw 0.
    w = winner(b)
    if w == "X":
        return 1
    if w == "O":
        return -1
    if " " not in b:
        return 0

    # Try every empty square; the next call belongs to the other player.
    next_player = "O" if player == "X" else "X"
    vals = [minimax(b[:i] + [player] + b[i+1:], next_player)
            for i in range(9) if b[i] == " "]

    # X maximizes the final score; O minimizes it.
    return max(vals) if player == "X" else min(vals)

### Step 2 — Score every legal move for X

In this position O already has two in the top row and threatens to win. We list X's legal moves and, for each, place an `X` there and ask minimax for the resulting value (with O to move next). The move with the highest value is X's best reply — `best` records which square that is.

In [ ]:
# O threatens the top row; it is X's turn to move.
board = ["O", "O", " ", " ", "X", " ", " ", " ", " "]

# Every empty square is a candidate move for X.
moves = [i for i in range(9) if board[i] == " "]

# Value of each move, assuming O replies optimally afterwards.
vals = [minimax(board[:i] + ["X"] + board[i+1:], "O") for i in moves]

# Index of the highest-valued move, mapped back to its board square.
best_index = int(max(range(len(vals)), key=lambda k: vals[k]))
best = moves[best_index]

### Step 3 — Plot the value of each move

Each bar is one candidate square and its minimax value. The best move is highlighted in green, the rest in red. The winning insight: only the square that blocks O's top-row threat avoids a forced loss — every other move lets O complete the line.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

# Highlight the best move in green, the rest in red.
cols = ["#7ee787" if i == best else "#ff7b72" for i in moves]
ax.bar([str(i) for i in moves], vals, color=cols)

# Label each bar with its minimax value.
for k, v in enumerate(vals):
    ax.text(k, v, str(v), ha="center", va="bottom")

ax.set_title("minimax value of each X move")
ax.set_xlabel("square")
plt.show()